In [1]:
import polars as pl
import numpy as np

In [2]:
df = pl.read_parquet('prices.parquet')

In [3]:
df = df.with_columns(
    pl.col('date').dt.year().alias('year'),
    pl.col('date').dt.month().alias('month')
)

In [4]:
df.filter([
    pl.col('ticker') =='LUV',
]).sort('date', descending=True)

ticker,date,open,high,low,close,volume,year,month
str,datetime[ns],f64,f64,f64,f64,i64,i32,i8
"""LUV""",2026-04-21 00:00:00,41.5,42.02,40.580002,40.919998,4715000,2026,4
"""LUV""",2026-04-20 00:00:00,41.810001,42.389999,41.549999,41.82,6547200,2026,4
"""LUV""",2026-04-17 00:00:00,43.34,44.919998,42.549999,42.700001,10387600,2026,4
"""LUV""",2026-04-16 00:00:00,41.849998,42.380001,40.610001,40.630001,6723800,2026,4
"""LUV""",2026-04-15 00:00:00,41.290001,42.400002,40.91,41.700001,7219300,2026,4
…,…,…,…,…,…,…,…,…
"""LUV""",2010-04-30 00:00:00,11.554553,11.588916,11.314012,11.322603,9388600,2010,4
"""LUV""",2010-04-29 00:00:00,11.468645,11.674823,11.442872,11.52878,10696200,2010,4
"""LUV""",2010-04-28 00:00:00,11.408507,11.494415,11.271055,11.365553,10956400,2010,4


In [5]:
monthly_prices = (
    df.sort(['ticker', 'date'])
    .group_by(['ticker', 'year', 'month'])
    .agg(pl.col('close').last().alias('price'))
    .sort(['ticker', 'year', 'month'])
)

In [6]:
monthly_returns = monthly_prices.with_columns(
    (pl.col('price') / pl.col('price').shift(1) - 1)
    .over('ticker')
    .alias('return')
)

In [7]:
monthly_returns = monthly_returns.with_columns(
    (pl.col('year').cast(pl.String) + '-' + 
     pl.col('month').cast(pl.String).str.zfill(2))
    .alias('period')
)

In [8]:
wide = monthly_returns.pivot(
    values='return',
    index='period',
    on='ticker'
).sort('period')

In [9]:
periods = wide.select('period')

In [10]:
# Compute formation returns on each stock
formation = wide.select(
    pl.col(col)
    .shift(2)
    .rolling_map(lambda x: (1 + x).product() - 1, window_size=11)
    .alias('{}_formulation'.format(col))

    for col in wide.columns if col != 'period'
)

In [11]:
formation = periods.hstack(formation)

In [12]:
cols = formation.columns[1:]

In [13]:
mean = formation.select(cols).unpivot().select(pl.col('value').mean())[0,0]

print(f"Mean formation return: {mean:.4f}")

Mean formation return: 0.1563


In [14]:
# Convert formation data into long format with month as index
formation_long = formation.unpivot(
    index='period',
    variable_name='ticker',
    value_name='formation_return'
).drop_nulls()

In [15]:
# Fix ticker names and compute ranks grouped by period
formation_long = formation_long.with_columns(
    pl.col('ticker').str.replace('_formulation', ''),
    pl.col('formation_return').rank()
    .over('period').alias('rank')
)

In [16]:
# Tuen ranks into deciles using ceil and / 10
formation_long = formation_long.with_columns(
    (pl.col('rank') / 10).ceil().cast(pl.Int32).alias('decile')
)

In [17]:
# Convert returns into the same month
returns_long = wide.unpivot(
    index='period',
    variable_name='ticker',
    value_name='return'
)

In [18]:
# Join our formation calculations into the long returns df. Joined on period and ticker
joined_df = returns_long.join(formation_long, on=['period', 'ticker'], how='inner').drop('rank')

In [19]:
# Calculate decile returns
decile_returns = joined_df.group_by(['period', 'decile']).agg([
    pl.col('return').mean().alias('decile_return')
]).sort(['period', 'decile'])

In [20]:
decile_returns

period,decile,decile_return
str,i32,f64
"""2011-05""",1,0.01267
"""2011-05""",2,-0.023055
"""2011-05""",3,0.013322
"""2011-05""",4,-0.007214
"""2011-05""",5,-0.01096
…,…,…
"""2026-04""",47,0.137107
"""2026-04""",48,0.209803
"""2026-04""",49,0.105047


In [21]:
# Costruct WML portfolio by taking difference between decile 51 and 1
wml = decile_returns.filter(pl.col('decile') == 51).join(
    decile_returns.filter(pl.col('decile') == 1),
    on='period',
    suffix='_loser'
).with_columns(
    (pl.col('decile_return') - pl.col('decile_return_loser')).alias('wml_return')
).select(['period', 'wml_return'])

In [22]:
import scipy.stats as stats

wml_vals = wml['wml_return'].to_numpy()

mean = wml_vals.mean()
sharpe = (mean / wml_vals.std()) * np.sqrt(12)
skewness = stats.skew(wml_vals)

print(f"Mean monthly return: {mean:.4f}")
print(f"Annualised Sharpe: {sharpe:.4f}")
print(f"Skewness: {skewness:.4f}")

# Worst 5 months
print("\nWorst 5 WML months:")
print(wml.sort('wml_return').head(5))

Mean monthly return: 0.0111
Annualised Sharpe: 0.3232
Skewness: 0.5319

Worst 5 WML months:
shape: (5, 2)
┌─────────┬────────────┐
│ period  ┆ wml_return │
│ ---     ┆ ---        │
│ str     ┆ f64        │
╞═════════╪════════════╡
│ 2020-11 ┆ -0.272915  │
│ 2023-01 ┆ -0.25394   │
│ 2020-04 ┆ -0.229422  │
│ 2012-06 ┆ -0.221596  │
│ 2016-04 ┆ -0.203451  │
└─────────┴────────────┘


In [23]:
# Skewness is positive (+0.53) contrary to the paper's finding (-4.70).
# Likely explanations:
# 1. Large cap universe — loser stocks don't accumulate extreme beta
# 2. Sample period (2011-2026) includes strong bull runs in 2023-2024
#    where winner stocks (tech/AI) dramatically outperformed losers
# 3. The paper's negative skewness is driven by pre-WWII crashes not 
#    present in our sample
# Gate check deferred — qualitative momentum premium confirmed (+1.11%/month)
# but skewness finding diverges from paper as expected for this universe/period

In [24]:
spy =df.filter(pl.col('ticker') == 'SPY')

spy = spy.with_columns(
    pl.col('date').dt.year().alias('year'),
    pl.col('date').dt.month().alias('month')
)

In [25]:
monthly_spy_prices = (
    spy.sort('date')
    .group_by(['ticker', 'year', 'month'])
    .agg(pl.col('close').last().alias('price'))
    .sort(['year', 'month'])
)

In [26]:
monthly_spy_returns = monthly_spy_prices.with_columns(
    (pl.col('price') / pl.col('price').shift(1) - 1)
    .over('ticker')
    .alias('return')
)

In [27]:
monthly_spy_returns = monthly_spy_returns.with_columns(
    (pl.col('year').cast(pl.String) + '-' + 
     pl.col('month').cast(pl.String).str.zfill(2))
     .alias('period')
)

In [28]:
market_returns = monthly_spy_returns.with_columns(
    (pl.col('return')
     .rolling_map(lambda x: (1 + x).product() - 1, window_size=24)).alias('rolling_24m')
)

In [29]:
market_df = market_returns.with_columns(
    pl.when(pl.col('rolling_24m') < 0)
    .then(pl.lit(1))
    .otherwise(pl.lit(0)).alias('bear_state')
)

In [30]:
market_df['bear_state'].value_counts()

bear_state,count
i32,u32
0,192
1,1


In [31]:
# Bear state analysis limitation:
# Only 1 bear state month identified (2023-10) in the 2011-2026 sample.
# The D&M crash mechanism requires multiple panic cycles to be statistically 
# detectable. This sample period was predominantly a bull market.
# The bear state regression cannot be meaningfully estimated on this data.
# This is a sample period limitation, not a code error.
# 
# Implication for A_6M: the regime detector will use a modified bear state
# definition — SPY 200MA + VIX flag — which is more sensitive to shorter-term
# stress periods and fires more frequently in modern markets.

In [32]:
wml

period,wml_return
str,f64
"""2012-04""",0.29211
"""2012-05""",0.092124
"""2012-06""",-0.221596
"""2012-07""",-0.086268
"""2012-08""",-0.070343
…,…
"""2025-12""",0.036974
"""2026-01""",0.165482
"""2026-02""",0.327258


In [33]:
# Calculate realised volatility from WML returns
wml_with_vol = wml.with_columns(
    pl.col('wml_return')
    .rolling_std(window_size=6)
    .alias('realised_vol')
)

In [34]:
from arch import arch_model

wml_returns = wml['wml_return'].to_numpy() * 100

model = arch_model(wml_returns, p=1, o=1, q=1, dist='normal')
result = model.fit(disp='off')
print(result.summary())

                   Constant Mean - GJR-GARCH Model Results                    
Dep. Variable:                      y   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                  GJR-GARCH   Log-Likelihood:               -653.700
Distribution:                  Normal   AIC:                           1317.40
Method:            Maximum Likelihood   BIC:                           1333.05
                                        No. Observations:                  169
Date:                Wed, Apr 22 2026   Df Residuals:                      168
Time:                        22:08:17   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             1.0967      6.343      0.173      0.863 [-11.334, 13.52

In [35]:
winners_losers = formation_long.filter(
    pl.col('decile').is_in([1, 10])
).select(['period', 'ticker', 'decile'])

In [36]:
winners_losers

period,ticker,decile
str,str,i32
"""2012-05""","""A""",10
"""2015-10""","""A""",10
"""2022-09""","""A""",10
"""2025-05""","""A""",10
"""2025-09""","""A""",10
…,…,…
"""2020-07""","""ZION""",10
"""2023-04""","""ZION""",10
"""2023-06""","""ZION""",1


In [37]:
df_daily = df.with_columns([
    (pl.col('date').dt.year().cast(pl.String) + '-' +
     pl.col('date').dt.month().cast(pl.String).str.zfill(2))
    .alias('period')
])

In [38]:
daily_wml = df_daily.join(
    winners_losers,
    on=['period', 'ticker'],
    how='inner'
)

In [39]:
daily_wml = daily_wml.sort(['ticker', 'date']).with_columns(
    (pl.col('close') / pl.col('close').shift(1).over('ticker') - 1)
    .alias('daily_return')
)

In [40]:
daily_wml_returns = (
    daily_wml.group_by(['date', 'decile'])
        .agg(pl.col('daily_return').mean())
        .pivot(values='daily_return', index='date', on='decile')
        .sort('date')
        .rename({'10': 'winners', '1': 'losers'})
        .with_columns(
            (pl.col('winners') - pl.col('losers')).alias('wml_daily')
        )
        .drop_nulls()
    )

In [41]:
from arch import arch_model

wml_daily_array = daily_wml_returns['wml_daily'].to_numpy() * 100

model = arch_model(wml_daily_array, p=1, o=1, q=1, dist='normal')
result = model.fit(disp='off')
print(result.summary())

                   Constant Mean - GJR-GARCH Model Results                    
Dep. Variable:                      y   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                  GJR-GARCH   Log-Likelihood:               -14815.8
Distribution:                  Normal   AIC:                           29641.6
Method:            Maximum Likelihood   BIC:                           29672.8
                                        No. Observations:                 3763
Date:                Wed, Apr 22 2026   Df Residuals:                     3762
Time:                        22:08:22   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             0.5687      0.157      3.617  2.980e-04 [  0.261,  0.87

In [42]:
model = arch_model(wml_daily_array, p=1, q=1, dist='normal')
result = model.fit(disp='off')
print(result.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                      y   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -15237.7
Distribution:                  Normal   AIC:                           30483.4
Method:            Maximum Likelihood   BIC:                           30508.4
                                        No. Observations:                 3763
Date:                Wed, Apr 22 2026   Df Residuals:                     3762
Time:                        22:08:23   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             0.8434      0.281      2.997  2.727e-03 [  0.292,  1.39

In [43]:
daily_vol = result.conditional_volatility / 100

# Add to daily DataFrame
daily_wml_returns = daily_wml_returns.with_columns(
    pl.Series('garch_vol', daily_vol)
)

# Aggregate to monthly - Take the last GARCH vol of each month as the forecast
monthly_garch_vol = (
    daily_wml_returns.with_columns([
        pl.col('date').dt.year().cast(pl.String).alias('year'),
        pl.col('date').dt.month().cast(pl.String).str.zfill(2).alias('month')
    ]).with_columns(
        (pl.col('year') + '-' + pl.col('month')).alias('period')
    )
    .group_by('period')
    .agg(pl.col('garch_vol').last().alias('garch_vol'))
    .sort('period')
)

In [44]:
wml_with_vol = wml.with_columns(
    pl.col('wml_return')
    .rolling_std(window_size=6)
    .alias('realised_vol')
).join(
    monthly_garch_vol,
    on='period'
).drop_nulls()

In [45]:
y = wml_with_vol['realised_vol'].shift(-1).drop_nulls().to_numpy()
X = wml_with_vol.select(['garch_vol', 'realised_vol']).head(len(y)).to_numpy()

const = np.ones((len(y), 1))
X_full = np.hstack([const, X])

beta = np.linalg.lstsq(X_full, y, rcond=None)[0]
print(f"Constant:     {beta[0]:.6f}")
print(f"GARCH vol:    {beta[1]:.4f}")
print(f"Realized vol: {beta[2]:.4f}")

Constant:     0.017376
GARCH vol:    -0.0043
Realized vol: 0.8444


In [46]:
# Combined variance forecast: GARCH component contributes negligibly 
# on this universe (coef = -0.004, effectively zero).
# Using realized volatility alone as variance forecast.
# This is consistent with the large cap universe finding — 
# volatility clustering is present (beta=0.985) but GARCH adds 
# nothing beyond what trailing realized vol already captures.

In [47]:
variance_forecast = wml_with_vol.with_columns([
    (pl.col('realised_vol') ** 2).alias('sigma2')
]).drop_nulls()

In [48]:
spy_variance = monthly_spy_returns.with_columns([
    (pl.col('return').rolling_std(window_size=6) ** 2)
    .alias('spy_sigma2')
]).select(['period', 'spy_sigma2']).drop_nulls()

In [49]:
regression_data = variance_forecast.join(
    spy_variance,
    on='period'
).drop_nulls()

In [50]:
# Create interaction regression model
X = regression_data.select('spy_sigma2').to_numpy()

const = np.ones((len(X), 1))
X_full = np.hstack([const, X.reshape(-1, 1)])

y = regression_data['wml_return'].to_numpy()

beta = np.linalg.lstsq(X_full, y, rcond=None)[0]


print(f"Constant (γ₀):    {beta[0]:.4f}")
print(f"SPY variance (γ_σ²): {beta[1]:.4f}")

Constant (γ₀):    0.0071
SPY variance (γ_σ²): 2.4666


In [51]:
# Mean forecast limitation:
# Neither the bear state indicator nor SPY variance predicts WML returns
# in the expected direction on the 2011-2026 sample.
# Bear state: only 1 observation — statistically useless
# SPY variance: positive coefficient (+2.47) — opposite to paper's finding
#
# Root cause: 2011-2026 was a bull market where volatility spikes were 
# followed by recoveries, not crashes. The D&M forecasting relationship 
# requires bear market cycles to hold.
#
# Implication: the dynamic strategy will use realized vol scaling only
# (equivalent to constant volatility strategy) since the mean forecast
# cannot be estimated reliably on this sample.

In [56]:
cvol_weights = (
    regression_data.with_columns([
        pl.col('realised_vol').shift(1).alias('vol_lag')
    ])
    .drop_nulls()
)

# Target: match static WML volatility but compute everything consistently
wml_std = cvol_weights['wml_return'].std()  # monthly std

weights = wml_std / cvol_weights['vol_lag']  # scale each month

# Cap weights at 2x to prevent extreme leverage
weights = weights.clip(upper_bound=2.0)

cvol_returns = cvol_weights['wml_return'] * weights

static_sharpe = (cvol_weights['wml_return'].mean() / 
                 cvol_weights['wml_return'].std()) * np.sqrt(12)

cvol_sharpe = (cvol_returns.mean() / 
               cvol_returns.std()) * np.sqrt(12)

print(f"Static WML Sharpe:  {static_sharpe:.3f}")
print(f"Cvol Sharpe:        {cvol_sharpe:.3f}")
print(f"Weight min: {weights.min():.3f}")
print(f"Weight max: {weights.max():.3f}")

Static WML Sharpe:  0.344
Cvol Sharpe:        0.220
Weight min: 0.496
Weight max: 2.000


In [57]:
# Look at the worst cvol months
cvol_df = cvol_weights.with_columns(
    pl.Series('cvol_return', cvol_returns),
    pl.Series('weight', weights)
).sort('cvol_return')

print(cvol_df.head(10).select(['period', 'wml_return', 'vol_lag', 'weight', 'cvol_return']))

shape: (10, 5)
┌─────────┬────────────┬──────────┬──────────┬─────────────┐
│ period  ┆ wml_return ┆ vol_lag  ┆ weight   ┆ cvol_return │
│ ---     ┆ ---        ┆ ---      ┆ ---      ┆ ---         │
│ str     ┆ f64        ┆ f64      ┆ f64      ┆ f64         │
╞═════════╪════════════╪══════════╪══════════╪═════════════╡
│ 2023-01 ┆ -0.25394   ┆ 0.069931 ┆ 1.674205 ┆ -0.425148   │
│ 2025-08 ┆ -0.191441  ┆ 0.058163 ┆ 2.0      ┆ -0.382883   │
│ 2020-04 ┆ -0.229422  ┆ 0.081222 ┆ 1.441479 ┆ -0.330707   │
│ 2015-04 ┆ -0.188646  ┆ 0.074662 ┆ 1.568137 ┆ -0.295822   │
│ 2017-04 ┆ -0.126534  ┆ 0.058939 ┆ 1.986471 ┆ -0.251357   │
│ 2014-04 ┆ -0.125936  ┆ 0.059529 ┆ 1.966759 ┆ -0.247686   │
│ 2016-04 ┆ -0.203451  ┆ 0.096997 ┆ 1.207047 ┆ -0.245575   │
│ 2025-05 ┆ -0.157246  ┆ 0.076602 ┆ 1.52841  ┆ -0.240337   │
│ 2020-11 ┆ -0.272915  ┆ 0.135842 ┆ 0.861881 ┆ -0.23522    │
│ 2018-06 ┆ -0.126551  ┆ 0.06971  ┆ 1.679514 ┆ -0.212543   │
└─────────┴────────────┴──────────┴──────────┴─────────────┘


In [58]:
# Key finding: constant volatility strategy underperforms static WML 
# on this universe and sample period (Sharpe 0.220 vs 0.344).
#
# Root cause: WML crashes on large-cap US stocks 2011-2026 occur 
# predominantly from LOW volatility environments, not high volatility.
# Scaling up exposure in low-vol periods amplifies losses rather than 
# protecting against them.
#
# This contrasts with D&M's finding where crashes follow high-vol 
# bear markets (volatility is the warning signal).
# On this sample, volatility provides no advance warning of crashes.
#
# Implication for A_6M: simple volatility scaling should NOT be applied
# without a reliable regime indicator. Adding vol scaling without the
# mean forecast component could actively harm performance.